# 11 — A multi-FA instrument, end to end, through the API

## Background: what the device is

This notebook flies a complete two-stage ion-optical instrument — an
**ion funnel** feeding a **hexapole ion guide** — entirely from Python,
with no dashboard in the loop. If you have never met either device:

**The ion funnel** is a stack of ring electrodes whose apertures shrink
along the axis. Adjacent rings carry radio-frequency (RF) voltages of
*opposite phase*, so an ion between them feels a rapidly oscillating
field. Averaged over an RF cycle, that oscillation acts like a repulsive
wall near the metal — an *effective (pseudo)potential* that rises where
the field is strong, i.e. near the ring edges — while a modest DC
voltage ladder from ring to ring pushes ions down the axis. At the
~Torr-to-mTorr pressures where funnels operate, collisions with the
buffer gas continually remove the ion's kinetic energy, so the packet
relaxes toward the axis as the apertures converge: the funnel *confines
and compresses* a diffuse ion cloud into a narrow beam (Kelly, Tolmachev,
Page, Tang & Smith, *Mass Spectrom. Rev.* **29**, 294 (2010)). The
collision model used throughout is the hard-sphere treatment of
Appelhans & Dahl (*Int. J. Mass Spectrom.* **244**, 1 (2005)); field
conventions follow Dahl (*Int. J. Mass Spectrom.* **200**, 3 (2000)).

**The hexapole guide** is six rod-like conductors around the axis with
alternating RF phase. Its effective potential is flat near the axis and
rises steeply toward the rods, so it transports ions over a long
distance with little mass discrimination. This instrument's variant
uses *tilted wires*, which add a weak axial push so ions do not stall
in the collision gas.

## Background: how ion_gym describes a multi-FA instrument

One JSON document *is* the instrument. Its parts:

* **`stages`** — an ordered list of field arrays (FAs). Each stage has a
  `name`, an inline **`spec`** (the same schema a single-device
  simulation uses: geometry, voltages, source, gas, integration), a
  **`pose`** (`offset_mm`, `rot_deg`) placing the stage's local frame in
  the shared world frame, and an **`exit`** plane where the flight hands
  the ion to the next stage.
* **`beam`** — where ions come from. `from_stage: funnel` means the
  packet is *produced by the funnel's own declared source* at load time
  (positions, velocities and per-ion m/z generated through the same code
  path as flying the funnel alone), then posed into the world frame.
  Editing the funnel's `source` block therefore edits the beam — that is
  the "explicit source control" this notebook demonstrates.
* **Seams** — at each exit plane the upstream field is dropped and the
  downstream field takes over. The handoff is only exact if the field is
  dead there, so the framework *measures* the worst-case field an ion can
  see at every seam and states it as a never-blocking advisory.

**The claim under test**: this funnel→hexapole instrument transports a
three-mass packet end to end, and its configuration — source parameters
and stage placement — can be inspected, edited, re-flown and re-drawn
from Python alone. Along the way you will see the geometry (multi-axis),
example trajectories, per-m/z arrival statistics, and what the seam
advisory does when a stage is physically moved.

## Named parameters

Every physics and display choice lives in this one cell, with units and
a one-line rationale each. Cells below consume these names — change a
value here and re-run from here.

In [ ]:
from pathlib import Path
from ion_gym.io.paths import repo_root

# ---- the instrument document (the deck of record; never edited in place)
INSTRUMENT_PATH = repo_root() / "examples" / "funnel_hexapole" / "instrument.json"

# ---- scratch folder for edited variants (load_assembly takes a path, so
#      each edited configuration is written out before loading; this dir
#      is excluded from release zips)
SCRATCH_DIR = repo_root() / "notebooks" / "out" / "nb11_variants"

# ---- reproducibility: the API default is fresh entropy per flight (by
#      design); a TEACHING document declares its seed so its numbers
#      reproduce. The seed is part of every figure's operating point.
SEED = 20260906            # dimensionless RNG seed

# ---- source edits demonstrated in section 4 (funnel source block)
N_IONS_PER_MZ = 25         # ions per mass; 25 x 3 masses = the deck's own
                           # 75-ion packet (declared flight parameters)
MZ_LIST = [200, 400, 600]  # Da; the deck's three demonstration masses
KE_LO_EV, KE_HI_EV = 0.1, 1.9   # eV; the deck's birth kinetic-energy
                                # window (uniform draw between them)
KE_HI_WIDE_EV = 6.0        # eV; section 4 widens the window to show how
                           # source edits propagate into transmission

# ---- pose edits demonstrated in section 7
GAP_SHIFT_MM = 2.0         # mm; 7a re-mounts the whole chain this far
                           # downstream (+z) -- stages AND exit planes
MISALIGN_X_MM = 1.0        # mm; 7b displaces the FUNNEL ALONE by this
MISALIGN_Y_MM = 1.5        # mm; (x, y) relative to the fixed hexapole,
                           # births pinned to the original axis (the
                           # chosen exhibit). Measured
                           # ladder behind the choice: (1, 1.5) mm
                           # transmits 36/75 with strong mass
                           # discrimination; (2, 3) mm is 0/75 and
                           # (4, 6) mm is 2/75 with a third of the
                           # packet dying on the displaced funnel's
                           # own rings.

# ---- recording (display-fidelity only; physics untouched). The shipped
#      hexapole deck records 100k rows at rec_every=10, which covers only
#      the first 1000 us of its 3000 us clock -- the loader itself warns
#      that traces truncate mid-flight. Raised to the funnel deck's own
#      400k so the drawn paths cover the ~1.2 ms measured flight.
HEX_MAX_RECORDS = 400_000  # rows; matches the sibling funnel deck

# ---- trajectory display
KEEP_TRACES = 25           # paths drawn (evenly strided across the
                           # packet); statistics always use ALL ions
TRACE_STYLE = dict(mode="lines", width=1.0, alpha=0.55, decim=3,
                   color_by="m/z")   # one colour per mass, with legend
FLY_COST_S_PER_ION = 0.2   # s/ion, measured warm in this container
                           # the first fly in a fresh
                           # kernel adds ~50-60 s of one-time JIT
print(f"instrument: {INSTRUMENT_PATH.relative_to(repo_root())}")
print(f"variants -> {SCRATCH_DIR.relative_to(repo_root())}")

## 1 — Read the document and see what an instrument *is*

The cell below loads the JSON as a plain dict — the entire configuration
surface — and prints each stage's pose and exit plus the beam contract.
Nothing is solved yet: this is the paperwork, and the paperwork is the
instrument.

In [ ]:
import copy, json

doc = json.loads(INSTRUMENT_PATH.read_text())
print(f"instrument {doc['name']!r} - {len(doc['stages'])} stages")
for st in doc["stages"]:
    pose = st.get("pose") or {}
    ex = st.get("exit") or {}
    g = st["spec"]["geometry"]
    print(f"  stage {st['name']!r}: {len(g['electrodes'])} electrodes, "
          f"coords {g['symmetry']['coords']}")
    print(f"    pose offset {pose.get('offset_mm')} mm, "
          f"rot {pose.get('rot_deg', [0, 0, 0])} deg")
    print(f"    exit: {ex.get('axis')} = {ex.get('value_mm')} mm"
          if ex else "    exit: none (final stage detects)")
print(f"beam: from_stage = {doc['beam']['from_stage']!r} at "
      f"m/z {doc['beam']['mz']:g} - the packet is GENERATED from that "
      f"stage's own source block at load time")
src = doc["stages"][0]["spec"]["source"]
print(f"funnel source: {src['n_ions']} ions/mass x m/z {src['mz_list']}, "
      f"{src['distribution']} r={src['r_mm']} mm, "
      f"KE {src['ke_lo']}-{src['ke_hi']} eV")

## 2 — A tiny variant-writer, and the working copy

`load_assembly` takes a *path* (one file is the instrument — a rule
that keeps geometry and configuration from drifting apart), so every
edited configuration is written to the scratch folder and loaded from
there. The working copy pins the declared `SEED` and raises the
hexapole's *recording* capacity (`max_records` — how many trajectory rows
are kept, not how the ion moves) so drawn paths cover the whole flight.
Voltages, clocks and geometry stay exactly as the deck of record states
them.

In [ ]:
def write_variant(document, name):
    """Write an edited instrument dict to the scratch dir; return its
    path for load_assembly. Named files, so every configuration flown in
    this notebook exists on disk and can be reloaded or shared."""
    SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
    p = SCRATCH_DIR / f"{name}.json"
    p.write_text(json.dumps(document, indent=1))
    return p

work = copy.deepcopy(doc)
work["stages"][0]["spec"]["source"]["seed"] = SEED          # declared
work["stages"][1]["spec"]["integration"]["max_records"] = HEX_MAX_RECORDS
BASELINE_PATH = write_variant(work, "baseline")
print(f"baseline variant written: {BASELINE_PATH.name} "
      f"(seed {SEED}; hexapole max_records {HEX_MAX_RECORDS:,})")

## 3 — See the instrument before flying it (multi-axis)

The framework's `assembly_overview` draws every stage in one world
frame, in three orthogonal panels with the axial coordinate (z)
horizontal. The dashed vertical lines are the seams — the funnel's exit
at z = 37.7 mm and the hexapole's at 88.55 mm. A stage misplaced along
any axis shows up in the panel that carries it; that is why a 3-D
assembly is never shown as a single projection.

In [ ]:
from ion_gym.io.spec_io import load_any_spec
from ion_gym.viz.viz_core import assembly_overview

def overview(document, trajs=None, subtitle=""):
    """Draw an instrument dict through the framework: stages posed in
    the world frame, seams marked, optional trajectories, and the
    operating point appended to the figure's own title."""
    stages, seams = [], []
    for st in document["stages"]:
        pose = st.get("pose") or {}
        stages.append((st["name"], load_any_spec(json.dumps(st["spec"])),
                       list(pose.get("offset_mm") or [0, 0, 0]),
                       list(pose.get("rot_deg") or [0, 0, 0])))
        ex = st.get("exit")
        if ex and ex.get("axis") in ("x", "y", "z"):
            seams.append({"axis": ex["axis"], "value_mm": ex["value_mm"],
                          "label": f"seam after {st['name']}"})
    fig = assembly_overview(stages, seams=seams, trajs=trajs,
                            traj_style=TRACE_STYLE, plane="all",
                            height=430)
    if subtitle:
        fig.layout.title.text += "  |  " + subtitle
    return fig

overview(work, subtitle=f"{doc['name']} as declared (seed {SEED})")

## 4 — Solve and fly the declared packet

`load_assembly` solves each stage's fields (the first ever solve of this
geometry takes ~80 s; afterwards the basis cache makes reloads a few
seconds — pose, source and integration edits reuse the cache because
none of them are geometry). `fly_packet` then flies all 75 ions and
reports arrivals, losses and per-ion records; the cost is quoted before
it is spent, and the loop streams progress. Note the `[SEAM ADVISORY]`:
the framework measured the worst field an ion can see at the funnel→
hexapole handoff and states it — loudly, quantified, never blocking.

In [ ]:
import numpy as np
import time
from ion_gym.physics.staged_flight import load_assembly, fly_packet
from ion_gym.progress import quote, track

t0 = time.time()
regions, beam = load_assembly(BASELINE_PATH)
print(f"load_assembly: {time.time() - t0:.1f} s "
      f"({len(regions)} regions solved or cache-hit; "
      f"{len(beam['ions'])} ions generated from "
      f"{beam['from_stage']!r}'s source)")

quote("baseline staged flight", n_items=len(beam["ions"]),
      per_item_s=FLY_COST_S_PER_ION,
      detail="first fly in a fresh kernel adds ~50-60 s one-time JIT")
out_base = fly_packet(regions, beam, keep_traces=KEEP_TRACES,
                      tracker=lambda rows: track(rows, "staged flight"))
print(f"arrived {out_base['n_arrived']}/{out_base['n']}  |  "
      f"losses by reason: {out_base['losses'] or 'none'}  |  "
      f"{out_base['n_traced']} paths kept for drawing")
print(f"headline note: {out_base.get('note', '(single-mass R would print here)')}")

### Per-m/z statistics

A time spread pooled across masses is not a resolution — the separation
between masses would masquerade as peak width — so the headline above
*refuses* a pooled R for this 3-mass packet, and the honest numbers are
per m/z. The table states n, arrivals, and the mean and spread of the
time of flight for each mass (operating point: the deck's own voltages,
seed above, funnel dt 2 ns / hexapole dt 1 ns).

In [ ]:
def per_mz_table(result, label):
    """Per-mass arrival statistics from the per-ion records. Prints n,
    arrivals, mean ToF and sd per m/z -- the honest multi-mass report."""
    rows = result["per_ion"]
    print(f"{label}  (n={result['n']}, seed {SEED}, "
          f"dt 2 ns funnel / 1 ns hexapole)")
    print(f"  {'m/z':>6} {'flown':>6} {'arrived':>8} "
          f"{'mean ToF (us)':>14} {'sd (us)':>9}")
    for m in sorted({p['mz'] for p in rows}):
        grp = [p for p in rows if p['mz'] == m]
        tof = np.array([p['t_us'] for p in grp
                        if p.get('arrived') and p.get('t_us') is not None])
        print(f"  {m:>6g} {len(grp):>6} {len(tof):>8} "
              + (f"{tof.mean():>14.1f} {tof.std(ddof=1):>9.2f}"
                 if len(tof) > 1 else f"{'-':>14} {'-':>9}"))

per_mz_table(out_base, "BASELINE - deck of record")

## 5 — Trajectories through the assembly

The same overview, now with the kept paths overlaid — one colour per
m/z, with a legend. Read the transport plane (z–x): ions are born in the
funnel's wide entrance disc, collapse toward the axis as the apertures
converge, cross the seam at 37.7 mm, and ride the hexapole's
pseudopotential channel to the detector plane at 88.55 mm. The drawn
subset (evenly strided) is stated in the title; statistics always use
every ion.

In [ ]:
overview(work, trajs=out_base["traces"],
         subtitle=f"baseline flight - {out_base['n_arrived']}/"
                  f"{out_base['n']} arrived (seed {SEED})")

## 6 — Edit the source, not the JSON on disk

Because `beam.from_stage = funnel`, the packet is regenerated from the
funnel's `source` block at every load — so editing that block *is*
editing the beam. Below, the birth kinetic-energy window widens from
0.1–1.9 eV to 0.1–6.0 eV while everything else (count, masses, radius,
seed) stays put. The question the measurement answers: does a hotter
birth population survive the funnel? The same handles set `n_ions`,
`mz_list`, the birth distribution and its radius — every field printed
in section 1.

In [ ]:
hot = copy.deepcopy(work)
hot_src = hot["stages"][0]["spec"]["source"]
hot_src["ke_lo"], hot_src["ke_hi"] = KE_LO_EV, KE_HI_WIDE_EV
hot_src["n_ions"] = N_IONS_PER_MZ           # explicit, from parameters
hot_src["mz_list"] = list(MZ_LIST)
HOT_PATH = write_variant(hot, "hot_source")

t0 = time.time()
regions_hot, beam_hot = load_assembly(HOT_PATH)
print(f"reload after a SOURCE edit: {time.time() - t0:.1f} s - the "
      f"solved fields cache-hit (a source block is not geometry)")
quote("hot-source staged flight", n_items=len(beam_hot["ions"]),
      per_item_s=FLY_COST_S_PER_ION)
out_hot = fly_packet(regions_hot, beam_hot, keep_traces=KEEP_TRACES,
                     tracker=lambda rows: track(rows, "hot-source flight"))
print(f"arrived {out_hot['n_arrived']}/{out_hot['n']}  |  "
      f"losses: {out_hot['losses'] or 'none'}")
per_mz_table(out_hot, f"HOT SOURCE - KE {KE_LO_EV}-{KE_HI_WIDE_EV} eV")
per_mz_table(out_base, "BASELINE (again, for the side-by-side)")

## 7 — Move the FAs: pose manipulation, the seam contract, and a loud failure

A stage's placement is its `pose`; moving hardware is arithmetic on
`offset_mm`. Two facts govern every move, and both are *measured* below:

1. **Stages must abut.** `fly_staged` imposes no drift model between
   regions — an ion leaving one stage's exit plane is handed to the next
   region at that same world point, so the next stage's solve box must
   own the previous exit plane. The legal rigid move of a connected
   instrument is therefore a move of the *chain*: below, BOTH stages and
   BOTH exit planes translate +2 mm together (re-mounting the whole
   instrument downstream), and the flight is unchanged.
2. **The seam advisory measures the UPSTREAM field dropped at the seam
   plane** — the funnel's own fringe at its own exit face — so it is
   *invariant* under moves of the downstream stage. Reducing it means
   changing the funnel side of the plane (its DC terminator ring is why
   the number is as small as it is), not sliding the hexapole.

`check_stage_clearance` guards every configuration: it refuses
interpenetrating declared metal by name before anything flies.

In [ ]:
from ion_gym.physics.staged_flight import (check_seam,
                                           check_stage_clearance,
                                           stage_shape_boxes_world,
                                           _pose_from)

# THE CONTRACT-PRESERVING MOVE: translate the whole chain +GAP_SHIFT_MM
# in z -- every stage offset and every exit plane together.
moved = copy.deepcopy(work)
for st in moved["stages"]:
    st["pose"]["offset_mm"][2] += GAP_SHIFT_MM
    if st.get("exit"):
        st["exit"]["value_mm"] += GAP_SHIFT_MM
MOVED_PATH = write_variant(moved, "chain_shifted")

# declared-metal clearance in the WORLD frame, both configurations
for tag, d in (("baseline", work), ("chain +%g mm" % GAP_SHIFT_MM, moved)):
    boxes = [(st["name"],
              stage_shape_boxes_world(load_any_spec(json.dumps(st["spec"])),
                                      _pose_from(st.get("pose"))))
             for st in d["stages"]]
    check_stage_clearance(boxes)     # raises, naming every pair, on overlap
    print(f"clearance [{tag}]: no declared-metal interpenetration")

t0 = time.time()
regions_mv, beam_mv = load_assembly(MOVED_PATH)
print(f"reload after a POSE edit: {time.time() - t0:.1f} s (cache-hit - "
      f"a pose is not geometry)")

# the seam, measured in both configurations -- expect INVARIANCE: the
# advisory quantifies the funnel's own field at its own exit face
for tag, regs in (("baseline", regions),
                  ("chain +%g mm" % GAP_SHIFT_MM, regions_mv)):
    ok, emax, rep = check_seam(regs[0], regs[1], float(doc["beam"]["mz"]))
    print(f"seam [{tag}]: worst-case |E| = {emax:.3g} V/mm "
          f"({'field-dead' if ok else 'LIVE - stated, never blocking'})")

In [ ]:
quote("chain-shifted staged flight", n_items=len(beam_mv["ions"]),
      per_item_s=FLY_COST_S_PER_ION)
out_mv = fly_packet(regions_mv, beam_mv, keep_traces=KEEP_TRACES,
                    tracker=lambda rows: track(rows, "chain-shift flight"))
print(f"arrived {out_mv['n_arrived']}/{out_mv['n']}  |  "
      f"losses: {out_mv['losses'] or 'none'}")
per_mz_table(out_mv, f"WHOLE CHAIN +{GAP_SHIFT_MM:g} mm - abutment kept")
overview(moved, trajs=out_mv["traces"],
         subtitle=f"whole instrument re-mounted +{GAP_SHIFT_MM:g} mm "
                  f"(exits now "
                  f"{moved['stages'][0]['exit']['value_mm']:g} / "
                  f"{moved['stages'][1]['exit']['value_mm']:g} mm; "
                  f"seed {SEED})")

### 7b — Misalign the funnel against the hexapole (births pinned)

Now a *relative* misalignment — the mis-mount that happens on a real
bench: the funnel bolted 1.0 mm off in x and 1.5 mm off in y while the
hexapole stays put, and the ion packet still born where the aligned
instrument put it. The API expresses the last part cleanly: a beam is
DATA, not welded to its instrument — `fly_packet(regions, beam)` takes
them independently, so the misaligned regions fly the BASELINE packet.

What to expect, from the physics: a funnel's whole job is delivering
ions to *its own* axis, so the exit beam is displaced by exactly the
funnel's displacement, and the hexapole receives it off-channel. The
measured consequence is the interesting part — read the per-m/z table
below before the prose after it.

In [ ]:
misal = copy.deepcopy(work)
misal["stages"][0]["pose"]["offset_mm"][0] += MISALIGN_X_MM   # funnel only;
misal["stages"][0]["pose"]["offset_mm"][1] += MISALIGN_Y_MM   # hexapole and
                                                              # exits untouched
MISAL_PATH = write_variant(misal, "funnel_misaligned")
regions_ms, _beam_moved = load_assembly(MISAL_PATH)
# BIRTHS PINNED: fly the BASELINE packet (section 4's `beam`) through the
# misaligned regions -- `_beam_moved` (regenerated on the displaced
# funnel) is deliberately unused, and saying so beats deleting the line.
import numpy as _np
_b = _np.array(beam["ions"])
print(f"flying the PINNED baseline packet: births centred "
      f"({_b[:, 0].mean():.2f}, {_b[:, 1].mean():.2f}) mm while the "
      f"funnel sits at (+{MISALIGN_X_MM:g}, +{MISALIGN_Y_MM:g}) mm")
quote("misaligned-funnel flight", n_items=len(beam["ions"]),
      per_item_s=FLY_COST_S_PER_ION)
out_ms = fly_packet(regions_ms, beam, keep_traces=KEEP_TRACES,
                    tracker=lambda rows: track(rows, "misaligned flight"))
print(f"arrived {out_ms['n_arrived']}/{out_ms['n']}  |  "
      f"losses: {out_ms['losses'] or 'none'}")
per_mz_table(out_ms, f"FUNNEL MISALIGNED (+{MISALIGN_X_MM:g}, "
                     f"+{MISALIGN_Y_MM:g}) mm vs fixed hexapole, "
                     f"births pinned")
overview(misal, trajs=out_ms["traces"],
         subtitle=f"funnel displaced (+{MISALIGN_X_MM:g}, "
                  f"+{MISALIGN_Y_MM:g}) mm against the fixed hexapole - "
                  f"non-concentric on purpose (seed {SEED})")

## Read-out: what each result means, and what failure would look like

**Section 3 (geometry).** Three orthogonal panels, z horizontal, seams
dashed. *If a pose were wrong* — say the hexapole offset by 7.8 mm in x —
the rod bundle would sit visibly beside the beam axis in the z–x panel
while z–y looked fine; that is exactly the class of misplacement a
single projection hides and this view exists to catch.

**Section 4 (baseline flight).** ~70 of 75 ions arriving matches the
instrument's flight of record, and the losses are *named*: a few births
land on funnel metal, a few ions leave the hexapole's solve box. The
`[SEAM ADVISORY]` (~42 V/mm worst-case at the funnel→hexapole plane) is
a *measured, bounded* defect statement, not an error — a worst case over
the whole plane, dominated by the outer fringe, while the transmitted
beam rides the bore of the DC terminator ring. *Had the advisory been
silent*, you should distrust the handoff, not celebrate: a seam nobody
measured is not a seam that is field-dead.

**Sections 4–5 (statistics and paths).** The pooled-R refusal on a
3-mass packet is the honest outcome; per-m/z tables carry the numbers.
Heavier ions arrive later — *if they did not*, the per-ion mass table
would be misaligned with the packet, which is precisely the defect the
framework refuses when the lengths disagree.

**Section 6 (source edit).** The measured answer: widening the birth-KE
window 0.1–1.9 → 0.1–6.0 eV left arrivals unchanged (70/75 both ways at
this n). That is the funnel doing its job — collisional cooling erases
the birth kinetic energy long before the exit, which is why funnels make
robust interfaces. The demonstration's real content is the *mechanism*:
the beam regenerated (watch the loss mix shuffle with the new draws)
because the edit landed on the `from_stage` FA's source block. *If the
outputs had been bit-identical*, the beam was NOT regenerated — check
that the edit reached the source stage and not a copy.

**Section 7 (pose moves).** 7a's axial chain translation flies
unchanged, with every seam, path and detection shifted +2 mm — poses
are manipulable, and for an *axial* move the exit planes travel with
their stages. The seam advisory is *invariant* (42.2 → 42.2 V/mm): it
measures the funnel's own dropped field at the plane, so only
funnel-side changes (its terminator ring, its ladder) move it. 7b's relative
mis-mount is the sharper lesson: the funnel faithfully delivers the
pinned packet to its OWN displaced axis, and the fixed hexapole receives
it 1.8 mm off-channel — transmission drops to 36/75 and becomes
strongly MASS-DEPENDENT (4/25 at m/z 200, 12/25 at 400, 20/25 at 600):
a transverse mis-mount does not merely attenuate, it masquerades as a
mass filter, shedding light ions preferentially. *The cliff, measured:*
at (+2, +3) mm nothing survives (0/75), and at (+4, +6) mm only 2/75
trickle through while a third of the packet dies on the displaced
funnel's own rings — the pinned births now sit under its metal. And
the axial failure mode from 7a's family still applies: move one stage
axially without its neighbour and ions hand off into a gap no solve box
owns, every one returning `left the solve box (kind 1)`. A stage's
placement is only as good as what the next stage can accept.

**Where to go next.** Every knob this notebook turned lives in the one
JSON document: add a stage, change `mz_list`, re-pose the chain, re-fly
— the same five calls (`load_assembly`, `fly_packet`, `check_seam`,
`check_stage_clearance`, `assembly_overview`) carry any configuration
you can write down.